# Implementation of an LSTM-Based Encoder–Decoder Architecture for Sequence-to-Sequence Learning.

In [1]:
import numpy as np
import tensorflow as tf
from tensorflow.keras.models import Model
from tensorflow.keras.layers import Input, LSTM, Dense

# Define problem parameters
seq_length = 5
vocab_size = 10      # Numbers 0-9
START_TOKEN = 10     # Token to indicate the start of decoding
END_TOKEN = 11       # Token to indicate the end of decoding
num_tokens = 12      # 0-9 plus START and END
latent_dim = 128     # Dimensionality of the encoding space
num_samples = 5000   # Number of training samples

In [2]:
# Generate Training Data
# Input: Random sequences of numbers 0-9
X = np.random.randint(0, vocab_size, size=(num_samples, seq_length))
# Target: Reversed sequences
Y = np.flip(X, axis=1)

# Initialize one-hot encoded arrays
encoder_input_data = np.zeros((num_samples, seq_length, num_tokens), dtype='float32')
decoder_input_data = np.zeros((num_samples, seq_length + 1, num_tokens), dtype='float32')
decoder_target_data = np.zeros((num_samples, seq_length + 1, num_tokens), dtype='float32')

for i, (input_seq, target_seq) in enumerate(zip(X, Y)):
    for t, num in enumerate(input_seq):
        encoder_input_data[i, t, num] = 1.0

    # Decoder input starts with START_TOKEN
    decoder_input_data[i, 0, START_TOKEN] = 1.0
    for t, num in enumerate(target_seq):
        decoder_input_data[i, t + 1, num] = 1.0
        decoder_target_data[i, t, num] = 1.0 # Target is ahead by one timestep

    # Decoder target ends with END_TOKEN
    decoder_target_data[i, len(target_seq), END_TOKEN] = 1.0

In [3]:
# Build the Training Model
encoder_inputs = Input(shape=(None, num_tokens), name="encoder_inputs")
encoder_lstm = LSTM(latent_dim, return_state=True, name="encoder_lstm")
encoder_outputs, state_h, state_c = encoder_lstm(encoder_inputs)
encoder_states = [state_h, state_c]

decoder_inputs = Input(shape=(None, num_tokens), name="decoder_inputs")
decoder_lstm = LSTM(latent_dim, return_sequences=True, return_state=True, name="decoder_lstm")
decoder_outputs, _, _ = decoder_lstm(decoder_inputs, initial_state=encoder_states)

decoder_dense = Dense(num_tokens, activation='softmax', name="decoder_dense")
decoder_outputs = decoder_dense(decoder_outputs)

model = Model([encoder_inputs, decoder_inputs], decoder_outputs)
model.compile(optimizer='adam', loss='categorical_crossentropy', metrics=['accuracy'])
model.summary()

Model: "functional"

┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ encoder_inputs      │ (None, None, 12)  │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ decoder_inputs      │ (None, None, 12)  │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ encoder_lstm (LSTM) │ [(None, 128),     │     72,192 │ encoder_inputs[0… │
│                     │ (None, 128),      │            │                   │
│                     │ (None, 128)]      │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ decoder_lstm (LSTM) │ [(None, None,     │     72,192 │ decoder_inputs[0… │
│                     │ 128), (None,      │            │ encoder_lstm[0][… │
│                     │ 128), (None,      │            │ encoder_lstm[0][… │
│                     │ 128)]             │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ decoder_dense       │ (None, None, 12)  │      1,548 │ decoder_lstm[0][… │
│ (Dense)             │                   │            │                   │
└─────────────────────┴───────────────────┴────────────┴───────────────────┘

 Total params: 145,932 (570.05 KB)

 Trainable params: 145,932 (570.05 KB)

 Non-trainable params: 0 (0.00 B)

In [4]:
# Train the model
print("Training the model...")
history = model.fit(
    [encoder_input_data, decoder_input_data],
    decoder_target_data,
    batch_size=64,
    epochs=30,
    validation_split=0.2,
    verbose=1
)
print("Training complete!")

Training the model...
Epoch 1/30
63/63 ━━━━━━━━━━━━━━━━━━━━ 5s 28ms/step - accuracy: 0.3014 - loss: 2.0660 - val_accuracy: 0.4463 - val_loss: 1.5031
Epoch 2/30
63/63 ━━━━━━━━━━━━━━━━━━━━ 1s 20ms/step - accuracy: 0.4738 - loss: 1.3229 - val_accuracy: 0.5145 - val_loss: 1.1856
Epoch 3/30
63/63 ━━━━━━━━━━━━━━━━━━━━ 1s 20ms/step - accuracy: 0.5412 - loss: 1.1025 - val_accuracy: 0.5927 - val_loss: 0.9911
Epoch 4/30
63/63 ━━━━━━━━━━━━━━━━━━━━ 2s 35ms/step - accuracy: 0.6910 - loss: 0.8642 - val_accuracy: 0.7845 - val_loss: 0.7115
Epoch 5/30
63/63 ━━━━━━━━━━━━━━━━━━━━ 2s 30ms/step - accuracy: 0.8570 - loss: 0.5731 - val_accuracy: 0.9195 - val_loss: 0.4318
Epoch 6/30
63/63 ━━━━━━━━━━━━━━━━━━━━ 1s 21ms/step - accuracy: 0.8624 - loss: 0.6265 - val_accuracy: 0.9520 - val_loss: 0.3352
Epoch 7/30
63/63 ━━━━━━━━━━━━━━━━━━━━ 1s 21ms/step - accuracy: 0.9758 - loss: 0.2611 - val_accuracy: 0.9842 - val_loss: 0.2081
Epoch 8/30
63/63 ━━━━━━━━━━━━━━━━━━━━ 2s 29ms/step - accuracy: 0.9914 - loss: 0.1733 - va

In [5]:
# Build the Inference Models (for prediction)
# Encoder inference model
encoder_model = Model(encoder_inputs, encoder_states)

# Decoder inference model
decoder_state_input_h = Input(shape=(latent_dim,))
decoder_state_input_c = Input(shape=(latent_dim,))
decoder_states_inputs = [decoder_state_input_h, decoder_state_input_c]

decoder_outputs_inf, state_h_inf, state_c_inf = decoder_lstm(
    decoder_inputs, initial_state=decoder_states_inputs)
decoder_states_inf = [state_h_inf, state_c_inf]
decoder_outputs_inf = decoder_dense(decoder_outputs_inf)

decoder_model = Model(
    [decoder_inputs] + decoder_states_inputs,
    [decoder_outputs_inf] + decoder_states_inf
)

In [6]:
# Function to reverse an array using the trained models
def reverse_array(input_array):
    # One-hot encode the input array
    input_seq = np.zeros((1, len(input_array), num_tokens), dtype='float32')
    for t, num in enumerate(input_array):
        input_seq[0, t, num] = 1.0

    # Encode the input as state vectors
    states_value = encoder_model.predict(input_seq, verbose=0)

    # Generate empty target sequence of length 1, populated with START_TOKEN
    target_seq = np.zeros((1, 1, num_tokens))
    target_seq[0, 0, START_TOKEN] = 1.0

    stop_condition = False
    decoded_sequence = []

    while not stop_condition:
        output_tokens, h, c = decoder_model.predict([target_seq] + states_value, verbose=0)

        # Sample a token
        sampled_token_index = np.argmax(output_tokens[0, -1, :])

        # Exit condition: either hit max length or find END_TOKEN
        if sampled_token_index == END_TOKEN or len(decoded_sequence) == seq_length:
            stop_condition = True
        else:
            decoded_sequence.append(sampled_token_index)

        # Update the target sequence (of length 1)
        target_seq = np.zeros((1, 1, num_tokens))
        target_seq[0, 0, sampled_token_index] = 1.0

        # Update states
        states_value = [h, c]

    return decoded_sequence

# Test the model
test_array = [3, 1, 4, 8, 5]
print(f"Original Array: {test_array}")
reversed_array = reverse_array(test_array)
print(f"Reversed Array (Predicted): {reversed_array}")

Original Array: [3, 1, 4, 8, 5]
Reversed Array (Predicted): [np.int64(5), np.int64(8), np.int64(4), np.int64(1), np.int64(3)]
